# Module 1 — Planting Window (maize, GHA)
Estimates the **planting dekad** per maize pixel: cue-fusion green-up (Sentinel-2 NDRE + S1 SAR + FPAR) for the main seasons, or CHIRPS rainfall onset (25/20 mm) for the short rains, then the inception-report **5+7 false-start gate**.

**Where to start:** put the `planting_pipeline` folder on your Google Drive, run the cells top-to-bottom, and approve the Drive-mount and Earth-Engine sign-in prompts.

## Setup

In [ ]:
!pip -q install earthengine-api geemap pandas geopandas 2>/dev/null
print('installed.')

In [ ]:
import ee
PROJECT="ee-manzikye"
try:
    ee.Initialize(project=PROJECT)
except Exception:
    ee.Authenticate(); ee.Initialize(project=PROJECT)
print("EE ready:", ee.String("ok").getInfo())

In [ ]:
from google.colab import drive; drive.mount("/content/drive")
import sys, os
PIPE_DIR="/content/drive/MyDrive/planting_pipeline"   # adjust if needed
assert os.path.isdir(PIPE_DIR), f"Upload planting_pipeline to Drive; not at {PIPE_DIR}"
sys.path.insert(0, PIPE_DIR); os.chdir(PIPE_DIR)
print("pipeline on path:", PIPE_DIR)

In [ ]:
# --- config + GEE-native map (geemap: built-in EE Layers panel, toggle + opacity) ---
COUNTRY="Kenya"      # "Kenya" | "Ethiopia"
SEASON ="Long rains" # "Long rains" | "Short rains" | "Meher"
YEAR=2024
S1_ORBIT="ASCENDING"   # S1B gone (2022) -> ASCENDING has coverage over Kenya
from run import GAUL_NAME
from src import zonal_aggregate as ZA
aoi = ZA.gaul_admin(ee, [GAUL_NAME[COUNTRY]], level=0).geometry()
aoi_run = ee.Geometry.Rectangle([34.4,-1.2,37.8,1.2])   # fast test box; use `aoi` for whole country
import geemap
try:
    from google.colab import output; output.enable_custom_widget_manager()  # needed for interactive geemap in Colab
except Exception:
    pass
def new_map(zoom=7):
    m = geemap.Map(add_google_map=False, basemap="SATELLITE")  # keyless Google tiles; native EE layer control
    m.centerObject(aoi_run, zoom)
    return m
def ee_layer(m, image, vis, name, shown=True, opacity=1.0):
    m.addLayer(ee.Image(image), vis, name, shown, opacity)  # appears in the Layers panel (toggle + opacity)
    return m
print(f"{COUNTRY} · {SEASON} · {YEAR} · S1 {S1_ORBIT}")

## Planting-window estimation

In [ ]:
# --- planting dekad (onset) — cue-fusion green-up (main seasons) or rainfall onset (short rains) ---
from src import (utils, s2_preprocess as S2, s1_preprocess as S1, fusion_phenometrics as FZ,
                 ltn as LTN, planting_date as PD, wrsi_feedback as WR)
from run import crop_mask_image
kc, soil = utils.load_crop_coeffs()
rows={(r['country'],r['season']):r for r in utils.viable_products(utils.load_calendar('config/season_calendar.csv')) if r['crop'].lower()=='maize'}
r=rows[(COUNTRY,SEASON)]; ss,se=utils.sos_window_dekads(r['sos_detection_window']); mask=crop_mask_image(ee,COUNTRY,'maize',None)
if SEASON=='Short rains':
    pet=WR.pet_dekadal(ee,aoi_run,YEAR); ch=WR.chirps_dekadal(ee,aoi_run,YEAR)
    planting=WR.wrsi_onset(ee,ch,ss,se,pet_ic=pet).updateMask(mask).toInt16()
else:
    s2=S2.build_s2_dekadal(ee,aoi_run,YEAR); s1=S1.build_s1_dekadal(ee,aoi_run,YEAR,orbit=S1_ORBIT); fpar=FZ.add_fpar_dekadal(ee,aoi_run,YEAR)
    g=FZ.build_fused_greenness(ee,s2,s1,fpar); ltn=LTN.build_ltn_prior(ee,aoi_run,ss,se)
    sos=FZ.detect_sos(ee,g,mask,ss,se,ltn_sos=ltn,ltn_pad=2); planting=PD.sos_to_planting(ee,sos,'maize').toInt16()
print('planting dekad computed for', COUNTRY, SEASON)

In [ ]:
# 5+7 false-start gate (green-up seasons; short rains already carries the 25/20 mm rule)
if SEASON!='Short rains':
    ok=WR.dryspell_false_start(ee,aoi_run,planting,YEAR,dk_lo=ss,dk_hi=se+2); planting=planting.updateMask(ok)
print('valid maize pixels:', planting.reduceRegion(ee.Reducer.count(),aoi_run,250,maxPixels=int(1e13)).get('planting_dekad').getInfo())

In [ ]:
M=new_map()
ee_layer(M, planting.clip(aoi_run), {'min':ss,'max':se+3,'palette':['440154','3b528b','21908d','5dc863','fde725']}, f'Planting dekad — {SEASON}')
M   # geemap renders its own GEE-native Layers panel (toggle + opacity slider) — no extra layer control needed

## Planting-window statistics
Distribution of the estimated planting dekad over maize area, an agreement (**skill**) score against the FEWS/FAO calendar window, and a per-admin table + ranked bar.

In [ ]:
# ============================================================
#  Planting-window STATISTICS  (run after the map cell)
#  distribution graph · calendar-agreement skill · per-admin table & ranked bar
# ============================================================
import numpy as np, pandas as pd, matplotlib.pyplot as plt
PIXEL_HA = 6.25                                   # area of one 250 m pixel
lab = utils.dekad_label                           # e.g. 9 -> "9\xb7Mar"

# ---- 1. AOI-wide planting-dekad distribution (area per dekad) --------------
hist = (planting.reduceRegion(ee.Reducer.frequencyHistogram(), aoi_run, 250,
        maxPixels=int(1e13)).get('planting_dekad').getInfo() or {})
H   = {int(round(float(k))): v for k, v in hist.items()}
dks = list(range(min(H) if H else ss, (max(H) if H else se+3) + 1))
cnt = np.array([H.get(d, 0) for d in dks], float)
area = cnt * PIXEL_HA
tot  = cnt.sum()
def wq(q):                                        # area-weighted quantile dekad
    c = np.cumsum(cnt); return int(np.array(dks)[np.searchsorted(c, tot*q)]) if tot else None
mode_dk = int(dks[int(np.argmax(cnt))]) if tot else None
mean_dk = float((cnt*np.array(dks)).sum()/tot) if tot else float('nan')

print(f"── {COUNTRY} \xb7 {SEASON} {YEAR} \xb7 planting-window statistics ──")
print(f"  maize area planted : {area.sum():,.0f} ha  ({int(tot):,} pixels)")
if tot:
    print(f"  modal dekad        : {lab(mode_dk)}")
    print(f"  median (p50) / mean: {lab(wq(0.5))}  /  {mean_dk:.1f}")
    print(f"  central 80% window : {lab(wq(0.1))}  →  {lab(wq(0.9))}   (spread {wq(0.9)-wq(0.1)} dekads)")
    inwin = area[[ss <= d <= se for d in dks]].sum()
    print(f"  SKILL — within FEWS/FAO calendar [{lab(ss)}–{lab(se)}]: {inwin/area.sum()*100:.0f}% of area")

# ---- 2. distribution bar chart (green = in calendar window, amber = outside)
fig, ax = plt.subplots(figsize=(8.4, 3.4))
ax.bar([lab(d) for d in dks], area/1000,
       color=['#3b7a57' if ss <= d <= se else '#c9a227' for d in dks])
ax.set_ylabel('maize area (000 ha)'); ax.set_xlabel('planting dekad')
ax.set_title(f'Planting-window distribution — {COUNTRY} {SEASON} {YEAR}')
ax.tick_params(axis='x', rotation=45)
for t in ax.get_xticklabels(): t.set_ha('right')
plt.tight_layout(); plt.show()

# ---- 3. per-admin planting window (GAUL level-1) --------------------------
admin = ZA.gaul_admin(ee, [GAUL_NAME[COUNTRY]], level=1).filterBounds(aoi_run)
feats = ZA.zonal_planting_stats(ee, planting, admin, scale=250).getInfo()['features']
rows = []
for f in feats:
    p = f['properties']; n = p.get('pd_count') or 0
    if n < 20: continue                           # drop near-empty units
    g = lambda k: lab(p[k]) if p.get(k) is not None else '—'
    rows.append(dict(Admin=p.get('ADM1_NAME','?'), _med=p.get('pd_p50'),
                     Modal=g('pd_mode'), Median=g('pd_p50'),
                     Early_p10=g('pd_p10'), Late_p90=g('pd_p90'),
                     Area_ha=round(n*PIXEL_HA)))
df = pd.DataFrame(rows).sort_values('_med').reset_index(drop=True)
display(df.drop(columns='_med'))

# ranked bar: median planting dekad by admin (earliest at top)
if len(df):
    fig, ax = plt.subplots(figsize=(7.5, max(2.4, 0.34*len(df))))
    ax.barh(df['Admin'], df['_med'], color='#3b528b'); ax.invert_yaxis()
    ax.set_xlabel('median planting dekad')
    for y, m in enumerate(df['_med']):
        if m is not None: ax.text(m, y, ' '+lab(int(m)), va='center', fontsize=8)
    ax.set_title(f'Median planting dekad by admin — {COUNTRY} {SEASON}')
    plt.tight_layout(); plt.show()

*Higher dekad = later planting. Export with `ee.batch.Export.image.toDrive(...)`; see `run.py` for batch runs.*